In [1]:
from pathlib import Path
import pandas as pd
from PyDI.io import load_parquet
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 25)
pd.set_option('display.max_colwidth', 100)

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
amazon_sample = load_parquet(
    DATA_DIR / "amazon_sample.parquet",
    name="amazon_sample"
)

goodreads_sample = load_parquet(
    DATA_DIR / "goodreads_sample.parquet",
    name="goodreads_sample"
)

metabooks_sample = load_parquet(
  DATA_DIR / "metabooks_sample.parquet",
  name="metabooks_sample"
)

In [3]:
from PyDI.io import load_csv

train_m2a = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="train_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2a = load_csv(
    MLDS_DIR / "test_MA.csv",
    name="test_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

train_m2g = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="train_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2g = load_csv(
    MLDS_DIR / "test_MG.csv",
    name="test_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

In [ ]:
from PyDI.entitymatching import StringComparator, NumericComparator
from PyDI.entitymatching import SortedNeighbourhoodBlocker

comparators_m2a = [
    StringComparator(column='title',similarity_function='cosine'),
    StringComparator(column='author',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='cosine', preprocess=str.lower),
    NumericComparator(column='publish_year',max_difference=1),
]
comparators_m2g=comparators_m2a+[
    NumericComparator(column="page_count", max_difference=10),
]



blocker_m2a = SortedNeighbourhoodBlocker(
    metabooks_sample, amazon_sample,
    key='title',
    window=10,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)



blocker_m2g = SortedNeighbourhoodBlocker(
    metabooks_sample, goodreads_sample,
    key='title',
    window=10,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

sorted_candidates_m2a = blocker_m2a.materialize()
sorted_candidates_m2g = blocker_m2g.materialize()

In [5]:
from PyDI.entitymatching import EntityMatchingEvaluator

# evaluate blocking: metabooks -> amazon
results_m2a = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=sorted_candidates_m2a,
    blocker=blocker_m2a,
    test_pairs=train_m2a,
    out_dir=BLOCK_EVAL_DIR
)
# evaluate blocking: metabooks -> goodreads
results_m2g = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=sorted_candidates_m2g,
    blocker=blocker_m2g,
    test_pairs=train_m2g,
    out_dir=BLOCK_EVAL_DIR
)

display(results_m2a)
display(results_m2g)



{'pair_completeness': 0.9390243902439024,
 'pair_quality': 0.0004259002674321809,
 'reduction_ratio': 0.9997048269387755,
 'total_candidates': 361587,
 'total_possible_pairs': 1225000000,
 'true_positives_found': 154,
 'total_true_pairs': 164,
 'evaluation_timestamp': '2025-11-12T15:42:49.857191',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

{'pair_completeness': 0.9230769230769231,
 'pair_quality': 0.00046433546451405506,
 'reduction_ratio': 0.9997257436734693,
 'total_candidates': 335964,
 'total_possible_pairs': 1225000000,
 'true_positives_found': 156,
 'total_true_pairs': 169,
 'evaluation_timestamp': '2025-11-12T15:42:59.366108',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

In [27]:
from PyDI.entitymatching import RuleBasedMatcher

matcher = RuleBasedMatcher()

# matching metabooks > amazon
correspondences_m2a = matcher.match(
    df_left=metabooks_sample,
    df_right=amazon_sample, 
    candidates=blocker_m2a,
    comparators=comparators_m2a,
    weights=[0.5, 0.3, 0.15,0.05], 
    threshold=0.7,
    id_column='id'
)

# matching metabooks > goodreads
correspondences_m2g = matcher.match(
    df_left=metabooks_sample,
    df_right=goodreads_sample, 
    candidates=blocker_m2g,
    comparators=comparators_m2g,

    weights=[0.45, 0.25, 0.15, 0.1, 0.05], 
    threshold=0.7,
    id_column='id'
)

In [28]:
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)
cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2a,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2g,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)


📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,19734,93.433076
1,3,582,2.755551
2,4,671,3.176933
3,5,59,0.279343
4,6,41,0.194120
5,7,13,0.061550
6,8,8,0.037877
7,9,2,0.009469
8,10,3,0.014204
9,12,3,0.014204



📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,4083,94.733179
1,3,205,4.756381
2,4,15,0.348028
3,5,3,0.069606
4,6,1,0.023202
5,7,1,0.023202
6,8,2,0.046404


In [29]:
from PyDI.entitymatching import MaximumBipartiteMatching, StableMatching

# We are using Maxmimum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
mbm_correspondences_m2a = clusterer.cluster(correspondences_m2a)
mbm_correspondences_m2g = clusterer.cluster(correspondences_m2g)


In [ ]:
correspondences_m2a.to_csv(CORR_DIR / "workflow_corr_m2a.csv", index=False)
correspondences_m2g.to_csv(CORR_DIR / "workflow_corr_m2g.csv", index=False)
mbm_correspondences_m2a = pd.read_csv(CORR_DIR / "workflow_corr_m2a.csv")
mbm_correspondences_m2g = pd.read_csv(CORR_DIR / "workflow_corr_m2g.csv")

In [30]:
all_correspondences = pd.concat([mbm_correspondences_m2a, mbm_correspondences_m2g], ignore_index=True)

len(all_correspondences)

26342

In [32]:
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
import pandas as pd

metabooks_sample.attrs["trust_score"] = 3
goodreads_sample.attrs["trust_score"] = 2
amazon_sample.attrs["trust_score"] = 1
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title', longest_string)
strategy.add_attribute_fuser('author', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('language', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('numratings', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres',union)

In [33]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_ml_sn_blocker.jsonl")

fused_rb_snblocker = engine.run(
    datasets=[amazon_sample, metabooks_sample, goodreads_sample],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False
)

print(f'Fused rows: {len(fused_rb_snblocker):,}')

Fused rows: 24,871


In [34]:
fused_rb_snblocker.to_parquet(PIPELINE_DIR / "fused_rb_snblocker.parquet")

In [35]:
fused_dataset = pd.read_parquet(PIPELINE_DIR / "fused_rb_snblocker.parquet")
fused_dataset["publish_year"] = fused_dataset["publish_year"].astype("Int16")
fused_dataset["page_count"] = fused_dataset["page_count"].astype("Int32")
golden_fused_dataset= pd.read_parquet(MLDS_DIR / "golden_fused_books.parquet")

In [36]:
from PyDI.fusion import tokenized_match, boolean_match,numeric_tolerance_match,set_equality_match

import numpy as np
import re, ast, numpy as np, pandas as pd


def categories_set_equal(a, b) -> bool:
    """Return True if a and b contain the same unique categories (order/type agnostic)."""
    def to_set(x):
        def items(v):
            # missing
            if v is None or (isinstance(v, float) and np.isnan(v)): return []
            # numpy array → recurse over elements
            if isinstance(v, np.ndarray): 
                out=[]; [out.extend(items(e)) for e in v.flatten()]; return out
            # python containers → recurse over elements
            if isinstance(v, (list, tuple, set)):
                out=[]; [out.extend(items(e)) for e in v]; return out
            # scalar/string: try parse stringified list; else split by delimiters
            s = str(v).strip()
            if s == "" or s.lower() in {"nan","none"}: return []
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)): return items(parsed)
            except Exception:
                pass
            return [p.strip() for p in re.split(r"[|,;/]", s) if p.strip()]
        return {it.lower() for it in items(x)}
    return to_set(a) == to_set(b)

strategy.add_evaluation_function("title", tokenized_match)
strategy.add_evaluation_function("author", tokenized_match)
strategy.add_evaluation_function("publisher", tokenized_match)
strategy.add_evaluation_function("publish_year", numeric_tolerance_match)
strategy.add_evaluation_function("numratings", numeric_tolerance_match)
strategy.add_evaluation_function("price", numeric_tolerance_match)
strategy.add_evaluation_function("page_count", numeric_tolerance_match)
strategy.add_evaluation_function("language", tokenized_match)
strategy.add_evaluation_function("genres", categories_set_equal)

In [37]:
from PyDI.fusion import DataFusionEvaluator
fused_dataset.drop_duplicates(subset='isbn_clean', keep='first',inplace=True)
# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused_dataset,
    fused_id_column='isbn_clean',
    gold_df=golden_fused_dataset,
    gold_id_column='isbn_clean',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.614
  macro_accuracy: 0.614
  num_evaluated_records: 38828
  num_evaluated_attributes: 9
  total_evaluations: 342014
  total_correct: 209997
  publish_year_accuracy: 0.614
  publish_year_count: 38708
  page_count_accuracy: 0.609
  page_count_count: 35411
  genres_accuracy: 0.597
  genres_count: 37448
  numratings_accuracy: 0.619
  numratings_count: 38828
  price_accuracy: 0.608
  price_count: 36440
  title_accuracy: 0.619
  title_count: 38828
  publisher_accuracy: 0.622
  publisher_count: 38828
  language_accuracy: 0.630
  language_count: 38695
  author_accuracy: 0.607
  author_count: 38828

Overall Accuracy: 61.4%
